<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Ch5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Aggregation & Window Function

**Part 1: Aggregation**
1. ORDER BY และลำดับการประมวลผล SQL
2. GROUP BY และ Aggregate Functions
3. WHERE เทียบกับ HAVING
4. Aggregate ขั้นสูง (FILTER, GROUPING SETS/ROLLUP/CUBE, STRING_AGG)

**Part 2: Window Functions**

5. พื้นฐาน Window Function (OVER, PARTITION BY)
6. Ranking Functions (ROW_NUMBER, RANK, DENSE_RANK, NTILE)
7. Aggregate Window & Frame (running total, moving average)
8. LAG/LEAD & Value Functions



## Setup

ใช้ **DuckDB** (in-memory) รันคำสั่ง SQL พร้อมข้อมูลตัวอย่าง (ห้องสมุด: `member`, `book`, `loan`)

In [1]:
# ติดตั้ง DuckDB (ถ้ารันบน Google Colab)
!pip install duckdb --quiet

In [2]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)

con = duckdb.connect(database=':memory:')

def run(sql):
    """รันคำสั่ง SQL แล้วคืนผลลัพธ์เป็น pandas DataFrame"""
    return con.execute(sql).df()

print("เชื่อมต่อ DuckDB (in-memory) สำเร็จ")

เชื่อมต่อ DuckDB (in-memory) สำเร็จ


### สร้างตารางและข้อมูลตัวอย่าง

ใช้ schema ห้องสมุดเดิม: `member` (สมาชิก), `book` (หนังสือ), `loan` (การยืม)

In [3]:
con.execute("""
CREATE TABLE member (
    member_id INTEGER,
    name VARCHAR
);

CREATE TABLE book (
    book_id INTEGER,
    title VARCHAR,
    category VARCHAR,
    price INTEGER,
    copies INTEGER
);

CREATE TABLE loan (
    loan_id INTEGER,
    book_id INTEGER,
    member_id INTEGER,
    loan_date DATE,
    status VARCHAR
);
""")

con.execute("""
INSERT INTO member VALUES
    (1, 'สมชาย'), (2, 'สมหญิง'), (3, 'มานะ'), (4, 'มานี'),
    (5, 'ปิติ'),   (6, 'ชูใจ'),   (7, 'วีระ'), (8, 'สมศรี');
""")

con.execute("""
INSERT INTO book VALUES
    (1, 'ฐานข้อมูลเบื้องต้น', 'เทคโนโลยี', 420, 3),
    (2, 'วิทยาการข้อมูล',     'เทคโนโลยี', 380, 2),
    (3, 'นิยายไทย',           'วรรณกรรม',  250, 5),
    (4, 'ประวัติศาสตร์',      'สังคม',     300, 0);
""")

con.execute("""
INSERT INTO loan VALUES
    (1,  1, 1, '2026-01-05', 'borrowed'),
    (2,  1, 2, '2026-01-20', 'borrowed'),
    (3,  2, 1, '2026-02-02', 'borrowed'),
    (4,  2, 2, '2026-02-10', 'borrowed'),
    (5,  1, 3, '2026-02-15', 'borrowed'),
    (6,  2, 3, '2026-03-01', 'borrowed'),
    (7,  1, 4, '2026-03-08', 'borrowed'),
    (8,  2, 4, '2026-03-20', 'returned'),
    (9,  3, 5, '2026-04-02', 'borrowed'),
    (10, 3, 6, '2026-04-15', 'borrowed'),
    (11, 3, 7, '2026-05-01', 'returned'),
    (12, 4, 8, '2026-05-10', 'returned');
""")

print("สร้างตาราง member, book, loan และเพิ่มข้อมูลตัวอย่างเรียบร้อย")

สร้างตาราง member, book, loan และเพิ่มข้อมูลตัวอย่างเรียบร้อย


In [4]:
run("SELECT * FROM book;")

,book_id,title,category,price,copies
0,1,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,3
1,2,วิทยาการข้อมูล,เทคโนโลยี,380,2
2,3,นิยายไทย,วรรณกรรม,250,5
3,4,ประวัติศาสตร์,สังคม,300,0


In [5]:
run("SELECT * FROM loan ORDER BY loan_date;")

,loan_id,book_id,member_id,loan_date,status
0,1,1,1,2026-01-05,borrowed
1,2,1,2,2026-01-20,borrowed
2,3,2,1,2026-02-02,borrowed
3,4,2,2,2026-02-10,borrowed
4,5,1,3,2026-02-15,borrowed
5,6,2,3,2026-03-01,borrowed
6,7,1,4,2026-03-08,borrowed
7,8,2,4,2026-03-20,returned
8,9,3,5,2026-04-02,borrowed
9,10,3,6,2026-04-15,borrowed


---
## 1. ORDER BY และลำดับการประมวลผล SQL

**ORDER BY** เป็นคำสั่งจัดเรียงผลลัพธ์ของ query ทำงาน**หลังสุด**ในลำดับการประมวลผลของ SQL (หลัง SELECT)

**ลำดับที่เราเขียน:** SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY

**ลำดับที่ฐานข้อมูลประมวลผลจริง:** FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

NOTE:
- สามารถเรียงหลายระดับ + ทิศทางต่างกันได้ เช่น `ORDER BY category ASC, price DESC`
- สามารถเรียงตาม expression หรือ alias ได้ (เพราะ ORDER BY ทำงานหลัง SELECT)
- ค่า NULL ถูกจัดลำดับต่างกันในแต่ละฐานข้อมูล — ถ้าสำคัญ ควรระบุ `NULLS FIRST` / `NULLS LAST` ให้ชัดเจนเสมอ

In [6]:
# ตัวอย่างที่ 1: เรียงตาม expression/alias ที่เพิ่งคำนวณ
# เรียงตามมูลค่าสต็อก (ราคา x เล่ม)
run("""
SELECT title, price, copies,
    price * copies AS stock_value
FROM book
ORDER BY stock_value DESC;
""")

,title,price,copies,stock_value
0,ฐานข้อมูลเบื้องต้น,420,3,1260
1,นิยายไทย,250,5,1250
2,วิทยาการข้อมูล,380,2,760
3,ประวัติศาสตร์,300,0,0


In [7]:
# ตัวอย่างที่ 2: เรียงหลายระดับ ทิศทางต่างกัน
run("""
SELECT title, category, price
FROM book
ORDER BY category ASC, price DESC;
""")

,title,category,price
0,นิยายไทย,วรรณกรรม,250
1,ประวัติศาสตร์,สังคม,300
2,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420
3,วิทยาการข้อมูล,เทคโนโลยี,380


ในภาษาไทยต้องระบุ collation เพิ่มเติม

การจัดเรียงข้อความมี 2 วิธีที่ต่างกัน

1. Binary/Code point collation (default) — เทียบตัวอักษรทีละตัวตามตำแหน่งในตาราง Unicode ตรงๆ ไม่สนใจภาษา ไม่สนใจการออกเสียง เป็นวิธีที่เร็วและง่าย
2. Linguistic collation — เทียบตามกฎการอ่านออกเสียงจริงของภาษานั้น ต้องมีตารางกฎเฉพาะภาษา คือ สิ่งที่ ICU collation (COLLATE th) ทำเพิ่มเติม ซึ่งจะ มีกฎพิเศษที่รู้จักสระนำ

ต้องระบุ collation เพิ่มเพราะ การเขียนกับการออกเสียงของภาษาไทยไม่ตรงกัน 100% เช่น กรณีมีสระนำ ซึ่ง default sort ของฐานข้อมูลออกแบบมาให้เร็วโดยเทียบแค่ code point ตรงๆ โดยที่ไม่มีความรู้เรื่องกฎภาษาศาสตร์อยู่ในตัว

ICU ย่อมาจาก International Components for Unicode — เป็นไลบรารีโอเพนซอร์สที่พัฒนาโดย Unicode Consortium (องค์กรเดียวกับที่กำหนดมาตรฐาน Unicode) ใช้สำหรับจัดการเรื่องการรองรับหลายภาษา (internationalization) ในซอฟต์แวร์ต่างๆ

In [8]:
run("""
SELECT title, category, price
FROM book
ORDER BY category COLLATE th ASC, price DESC;
""")

,title,category,price
0,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420
1,วิทยาการข้อมูล,เทคโนโลยี,380
2,นิยายไทย,วรรณกรรม,250
3,ประวัติศาสตร์,สังคม,300


In [9]:
# ตัวอย่างที่ 3: ควบคุมตำแหน่งของ NULL ด้วย NULLS LAST
con.execute("ALTER TABLE book ADD COLUMN price_nullable INTEGER")
con.execute("UPDATE book SET price_nullable = price")
con.execute("INSERT INTO book VALUES (5, 'หนังสือใหม่', 'เทคโนโลยี', NULL, 0, NULL)")

result = run("""
SELECT title, price_nullable AS price
FROM book
ORDER BY price_nullable DESC NULLS LAST;
""")
con.execute("DELETE FROM book WHERE book_id = 5")   # ลบแถวทดลองออกไม่ให้กระทบหัวข้อถัดไป
con.execute("ALTER TABLE book DROP COLUMN price_nullable")
result

,title,price
0,ฐานข้อมูลเบื้องต้น,420
1,วิทยาการข้อมูล,380
2,ประวัติศาสตร์,300
3,นิยายไทย,250
4,หนังสือใหม่,<NA>


### แบบฝึกหัดที่ 1 (ORDER BY)

**โจทย์:** ให้เขียน query แสดง `title`, `category`, `price` ของหนังสือทั้งหมด โดยเรียงตามหมวด (`category`) จาก ฮ→ก ก่อน แล้วภายในหมวดเดียวกันให้เรียงราคาจากน้อยไปมาก

**NOTE:** ใช้ `ORDER BY` สองคอลัมน์ กำหนดทิศทางแยกกันคนละคอลัมน์ได้

In [11]:
# เขียนโค้ดตรงนี้
run("""
  SELECT title, category, price FROM book
  ORDER BY category COLLATE th DESC

""")

,title,category,price
0,ประวัติศาสตร์,สังคม,300
1,นิยายไทย,วรรณกรรม,250
2,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420
3,วิทยาการข้อมูล,เทคโนโลยี,380


In [ ]:
# ตัวอย่างผลลัพธ์

,title,category,price
0,ประวัติศาสตร์,สังคม,300
1,นิยายไทย,วรรณกรรม,250
2,วิทยาการข้อมูล,เทคโนโลยี,380
3,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420


---
## 2. GROUP BY และ Aggregate Functions

**GROUP BY** ใช้จัดกลุ่มแถวที่มีค่าเหมือนกัน แล้ว**ยุบแต่ละกลุ่มให้เหลือ 1 แถว** พร้อมคำนวณค่าสรุปด้วย **Aggregate Function** (COUNT, SUM, AVG, MIN, MAX)

**NOTE:** ทุกคอลัมน์ใน SELECT ที่ไม่ได้อยู่ใน aggregate function ต้องอยู่ใน GROUP BY ด้วย มิฉะนั้นจะ error เพราะฐานข้อมูลไม่รู้ว่าจะเลือกค่าไหนมาเป็นตัวแทนของกลุ่ม

In [ ]:
# ตัวอย่างที่ 1: GROUP BY พื้นฐาน
run("""
SELECT category,
    COUNT(*) AS n_books,
    ROUND(AVG(price), 0) AS avg_price,
    MIN(price) AS min_p,
    MAX(price) AS max_p,
    SUM(copies) AS total_copies
FROM book
GROUP BY category;
""")

,category,n_books,avg_price,min_p,max_p,total_copies
0,เทคโนโลยี,2,400.0,380,420,5.0
1,วรรณกรรม,1,250.0,250,250,5.0
2,สังคม,1,300.0,300,300,0.0


In [ ]:
# ตัวอย่างที่ 2: ทำไม query นี้ error -- title ไม่อยู่ใน GROUP BY และไม่ใช่ aggregate
try:
    run("""
    SELECT title, category, COUNT(*)
    FROM book
    GROUP BY category;
    """)
except Exception as e:
    print("Error (ตามที่คาดไว้):")
    print(e)

Error (ตามที่คาดไว้):
Binder Error: column "title" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(title)" if the exact value of "title" is not important.

LINE 2:     SELECT title, category, COUNT(*)
                   ^


In [ ]:
# ทดลองเพิ่ม title ไปที่ group by
run("""
    SELECT title, category, COUNT(*)
    FROM book
    GROUP BY category, title;
""")

,title,category,count_star()
0,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,1
1,ประวัติศาสตร์,สังคม,1
2,วิทยาการข้อมูล,เทคโนโลยี,1
3,นิยายไทย,วรรณกรรม,1


In [ ]:
# ทดลองเพิ่มหนังสือ เพื่อให้ดูว่า count ทำงานที่อะไร เพิ่ม order by เข้าไปด้วย
# เพิ่มหนังสือชื่อซ้ำกับเล่มที่มีอยู่แล้ว (book_id=5 ชื่อ+หมวดเดียวกับ book_id=1)
con.execute("INSERT INTO book VALUES (5, 'ฐานข้อมูลเบื้องต้น', 'เทคโนโลยี', 420, 1)")

result = run("""
    SELECT title, category, COUNT(*)
    FROM book
    GROUP BY category, title
    ORDER BY category COLLATE th ASC;
""")

con.execute("DELETE FROM book WHERE book_id = 5")   # ลบแถวทดลองออก ไม่ให้กระทบหัวข้อถัดไป
result

,title,category,count_star()
0,วิทยาการข้อมูล,เทคโนโลยี,1
1,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,2
2,นิยายไทย,วรรณกรรม,1
3,ประวัติศาสตร์,สังคม,1


In [ ]:
# ตัวอย่างที่ 3: COUNT(*) เทียบ COUNT(DISTINCT ...)
# หาจำนวนสมาชิกไม่ซ้ำที่ยืมแต่ละหมวด
run("""
SELECT b.category,
    COUNT(*) AS total_loans,
    COUNT(DISTINCT l.member_id) AS unique_members
FROM loan l
JOIN book b ON l.book_id = b.book_id
GROUP BY b.category;
""")

,category,total_loans,unique_members
0,เทคโนโลยี,8,4
1,วรรณกรรม,3,3
2,สังคม,1,1


### แบบฝึกหัดที่ 2 (GROUP BY)

**โจทย์:** ให้เขียน query แสดง หมวดหมู่หนังสือ จำนวนหนังสือ และราคาเฉลี่ย ของแต่ละหมวด โดยให้จัดเรียงตามราคาเฉลี่ยของแต่ละหมวดจากมากไปน้อย

**NOTE:** ใช้ `GROUP BY category`, ใช้ `COUNT(*)` และ `AVG(price)`, `ORDER BY` ค่าเฉลี่ย

In [16]:
# เขียนโค้ดตรงนี้
run("""
    SELECT category, COUNT(*) AS n_books, AVG(price) AS avg_price
    FROM book
    GROUP BY category
    ORDER BY avg_price DESC
""")

,category,n_books,avg_price
0,เทคโนโลยี,2,400.0
1,สังคม,1,300.0
2,วรรณกรรม,1,250.0


In [ ]:
# ตัวอย่างผลลัพธ์

,category,n_books,avg_price
0,เทคโนโลยี,2,400.0
1,สังคม,1,300.0
2,วรรณกรรม,1,250.0


---
## 3. WHERE เทียบกับ HAVING

ทั้งคู่ทำหน้าที่กรองข้อมูลเช่นเดียวกัน แต่ทำงานคนละจังหวะในลำดับการประมวลผล:

| | WHERE | HAVING |
|---|---|---|
| ทำงานตอนไหน | ก่อน GROUP BY (กรองทีละแถว) | หลัง GROUP BY (กรองทีละกลุ่ม) |
| ใช้กับ aggregate ได้ไหม | ❌ ไม่ได้ | ✅ ได้ |
| ตัวอย่าง | `WHERE price >= 300` | `HAVING COUNT(*) >= 3` |

**NOTE:** เงื่อนไขกับ "ค่าในแถว" → WHERE · เงื่อนไขกับ "ผลสรุปของกลุ่ม" → HAVING

In [ ]:
# ตัวอย่าง: ใช้ WHERE และ HAVING ร่วมกัน
# หาหมวดที่มีการยืมสถานะ borrowed ตั้งแต่ 3 ครั้งขึ้นไป
run("""
SELECT b.category, COUNT(*) AS loans
FROM loan l
JOIN book b ON l.book_id = b.book_id
WHERE l.status = 'borrowed'
GROUP BY b.category
HAVING COUNT(*) >= 3
ORDER BY loans DESC;
""")
#FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

,category,loans
0,เทคโนโลยี,7


### แบบฝึกหัดที่ 3 (WHERE vs HAVING)

**โจทย์:** ให้หาหมวดที่มีราคาเฉลี่ยตั้งแต่ 300 บาทเป็นต้นไป และจัดเรียงลำดับจากราคาเฉลี่ยจากมากไปน้อย

In [22]:
# เขียนโค้ดตรงนี้
run("""
    SELECT category, AVG(price) AS avg_price
    FROM book
    GROUP BY category
    HAVING avg_price >= 300
    ORDER BY avg_price DESC
""")

,category,avg_price
0,เทคโนโลยี,400.0
1,สังคม,300.0


In [ ]:
# ตัวอย่างผลลัพธ์

,category,avg_price
0,เทคโนโลยี,400.0
1,สังคม,300.0


---
## 4. Aggregate ขั้นสูง

เทคนิคที่ต่อยอดจาก GROUP BY ธรรมดา ใช้สร้าง**รายงานสรุปหลายระดับ/หลายมิติในคำสั่งเดียว**

- **FILTER** — จำกัดขอบเขตของ aggregate function`
- **ROLLUP** — สรุปแบบไล่ลำดับชั้น (ยุบจากขวาไปซ้าย)
- **CUBE** — สรุปทุกชุดผสมที่เป็นไปได้
- **GROUPING SETS** — ระบุชุดสรุปเองอย่างอิสระ
- **GROUPING()** — บอกว่า NULL ในแถวนั้นเป็น "แถวสรุป" หรือ NULL จริงจากข้อมูล (คืน 1 หรือ 0)
- **STRING_AGG** — รวมค่าข้อความของกลุ่มเป็นสตริงเดียว

In [ ]:
# ตัวอย่างที่ 1: FILTER -- นับแบบมีเงื่อนไขในกลุ่มเดียว
# ต้องการสรุปสถานะการยืมของแต่ละหมวดหนังสือ
run("""
SELECT b.category,
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE l.status = 'borrowed') AS borrowed,
    COUNT(*) FILTER (WHERE l.status = 'returned') AS returned
FROM loan l
JOIN book b ON l.book_id = b.book_id
GROUP BY b.category
ORDER BY category COLLATE th ASC;
""")

,category,total,borrowed,returned
0,เทคโนโลยี,8,7,1
1,วรรณกรรม,3,2,1
2,สังคม,1,0,1


In [ ]:
# ตัวอย่างที่ 2: ROLLUP + GROUPING() -- รายงานหลายระดับพร้อม grand total
# ต้องการแสดงรายงานยอดยืมแยกตามหมวด พร้อมยอดรวมทั้งหมด (grand total) ในตารางเดียว
run("""
SELECT
    b.category,
    CASE WHEN GROUPING(b.category) = 1 THEN 'ทุกหมวด' ELSE b.category END AS category_g,
    COUNT(*) AS n
FROM loan l
JOIN book b ON l.book_id = b.book_id
GROUP BY ROLLUP(b.category)
ORDER BY GROUPING(b.category), b.category COLLATE th ASC;
""")

,category,category_g,n
0,เทคโนโลยี,เทคโนโลยี,8
1,วรรณกรรม,วรรณกรรม,3
2,สังคม,สังคม,1
3,None,ทุกหมวด,12


In [ ]:
# ตัวอย่างที่ 3: CUBE -- ทุกชุดผสมที่เป็นไปได้ (มี subtotal ตาม status ด้วย ซึ่ง ROLLUP ไม่มี)
# ต้องการแสดงรายงานยอดยืมแยกทุกหมวดทุกสถานะ
run("""
SELECT
    b.category,
    l.status,
    CASE WHEN GROUPING(b.category) = 1 THEN 'ทุกหมวด' ELSE b.category END AS category_g,
    CASE WHEN GROUPING(l.status)   = 1 THEN 'ทุกสถานะ' ELSE l.status END   AS status_g,
    COUNT(*) AS n
FROM loan l
JOIN book b ON l.book_id = b.book_id
GROUP BY CUBE(b.category, l.status)
ORDER BY GROUPING(b.category), GROUPING(l.status), category COLLATE th ASC, status;
""")

,category,status,category_g,status_g,n
0,เทคโนโลยี,borrowed,เทคโนโลยี,borrowed,7
1,เทคโนโลยี,returned,เทคโนโลยี,returned,1
2,วรรณกรรม,borrowed,วรรณกรรม,borrowed,2
3,วรรณกรรม,returned,วรรณกรรม,returned,1
4,สังคม,returned,สังคม,returned,1
5,เทคโนโลยี,None,เทคโนโลยี,ทุกสถานะ,8
6,วรรณกรรม,None,วรรณกรรม,ทุกสถานะ,3
7,สังคม,None,สังคม,ทุกสถานะ,1
8,None,borrowed,ทุกหมวด,borrowed,9
9,None,returned,ทุกหมวด,returned,3


In [ ]:
run("""
select * from book
""")

,book_id,title,category,price,copies
0,1,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,3
1,2,วิทยาการข้อมูล,เทคโนโลยี,380,2
2,3,นิยายไทย,วรรณกรรม,250,5
3,4,ประวัติศาสตร์,สังคม,300,0


In [ ]:
# ตัวอย่างที่ 4: STRING_AGG -- รวมรายชื่อหนังสือของแต่ละหมวดเป็นบรรทัดเดียว
# โดยรายชื่อหนังสือต่อหมวดให้เรียงตามราคาจากมากไปน้อย
run("""
SELECT category,
    STRING_AGG(title, ', ' ORDER BY price DESC) AS books
FROM book
GROUP BY category;
""")

,category,books
0,เทคโนโลยี,"ฐานข้อมูลเบื้องต้น, วิทยาการข้อมูล"
1,วรรณกรรม,นิยายไทย
2,สังคม,ประวัติศาสตร์


### แบบฝึกหัดที่ 4 (Aggregate ขั้นสูง)

**โจทย์:** ให้แสดงรายงานยอดการยืมในแต่ละหมวดและยอดยืมรวมต่อสถานะ

In [34]:
# เขียนโค้ดตรงนี้
run("""
SELECT
    CASE WHEN GROUPING(b.category) = 1 THEN 'ทุกหมวด' ELSE b.category END AS category,
    CASE WHEN GROUPING(l.status) = 1 THEN 'ทุกสถานะ' ELSE l.status END AS status,
    COUNT(*) AS n
FROM loan l
JOIN book b ON l.book_id = b.book_id
GROUP BY GROUPING SETS(b.category, l.status)

""")

,category,status,n
0,เทคโนโลยี,ทุกสถานะ,8
1,วรรณกรรม,ทุกสถานะ,3
2,สังคม,ทุกสถานะ,1
3,ทุกหมวด,borrowed,9
4,ทุกหมวด,returned,3


In [ ]:
# ตัวอย่างผลลัพธ์

,category,status,n
0,เทคโนโลยี,ทุกสถานะ,8
1,วรรณกรรม,ทุกสถานะ,3
2,สังคม,ทุกสถานะ,1
3,ทุกหมวด,borrowed,9
4,ทุกหมวด,returned,3


---
# Part 2: Window Functions

---
## 5. พื้นฐาน Window Function

**Window Function** คือฟังก์ชันที่**คำนวณข้ามหลายแถว แต่ไม่ยุบแถวทิ้ง** — ต่างจาก aggregate function ธรรมดาที่ใช้กับ GROUP BY (ยุบเหลือ 1 แถวต่อกลุ่ม) window function จะเก็บทุกแถวไว้ครบ แล้วเติมผลคำนวณลงเป็นคอลัมน์ใหม่

**โครงสร้าง:** `ฟังก์ชัน() OVER (PARTITION BY ... ORDER BY ...)`

- **`OVER ()`** — คำนวณจากทั้งตาราง เติมค่าเดียวกันทุกแถว
- **`PARTITION BY column`** — แบ่งเป็นกลุ่มย่อยก่อนคำนวณ (เหมือน GROUP BY แต่ไม่ยุบ)

ทดสอบการเทียบคำสั่ง GROUP BY (ยุบแถว) กับ Window Function (ไม่ยุบแถว) ในโจทย์หาค่าเฉลี่ยหนังสือแต่ละหมวด

In [ ]:
print("-- GROUP BY: ยุบแถว --")
run("""
SELECT category, AVG(price) AS avg_price
FROM book
GROUP BY category;
""")

-- GROUP BY: ยุบแถว --


,category,avg_price
0,เทคโนโลยี,400.0
1,วรรณกรรม,250.0
2,สังคม,300.0


In [ ]:
print("-- Window Function: ไม่ยุบแถว --")
run("""
SELECT title, category, price,
    AVG(price) OVER (PARTITION BY category) AS avg_cat
FROM book;
""")

-- Window Function: ไม่ยุบแถว --


,title,category,price,avg_cat
0,ประวัติศาสตร์,สังคม,300,300.0
1,นิยายไทย,วรรณกรรม,250,250.0
2,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,400.0
3,วิทยาการข้อมูล,เทคโนโลยี,380,400.0


- **`OVER ()`** — คำนวณจากทั้งตาราง เติมค่าเดียวกันทุกแถว

In [ ]:
# ตัวอย่างการใช้ OVER()
# ต้องการหาว่าแต่ละเล่มราคาสูง/ต่ำกว่าค่าเฉลี่ยของหนังสือทั้งหมดเท่าไหร่
run("""
SELECT title, price,
    ROUND(AVG(price) OVER (), 0) AS avg_all,
    price - ROUND(AVG(price) OVER (), 0) AS diff
FROM book;
""")

,title,price,avg_all,diff
0,ฐานข้อมูลเบื้องต้น,420,338.0,82.0
1,วิทยาการข้อมูล,380,338.0,42.0
2,นิยายไทย,250,338.0,-88.0
3,ประวัติศาสตร์,300,338.0,-38.0


- **`PARTITION BY column`** — แบ่งเป็นกลุ่มย่อยก่อนคำนวณ (เหมือน GROUP BY แต่ไม่ยุบ)

In [ ]:
# ตัวอย่างการใช้ PARTITION BY
# ต้องการหาราคาแต่ละเล่ม เทียบกับราคาเฉลี่ยของหมวดตัวเอง
run("""
SELECT title, category, price,
    ROUND(AVG(price) OVER (PARTITION BY category), 0) AS avg_cat
FROM book
""")

,title,category,price,avg_cat
0,ประวัติศาสตร์,สังคม,300,300.0
1,นิยายไทย,วรรณกรรม,250,250.0
2,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,400.0
3,วิทยาการข้อมูล,เทคโนโลยี,380,400.0


### แบบฝึกหัดที่ 5 (Window Function)

**โจทย์:** ให้เขียน query แสดง `title`, `category`, `price` พร้อม ค่าเฉลี่ยราคาของหมวดตนเองจากน้อยไปมาก และส่วนต่างจากค่าเฉลี่ยในหมวดตนเอง


In [ ]:
# เขียนโค้ดตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์

,title,category,price,avg_cat,diff_from_cat
0,วิทยาการข้อมูล,เทคโนโลยี,380,400.0,-20.0
1,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,400.0,20.0
2,นิยายไทย,วรรณกรรม,250,250.0,0.0
3,ประวัติศาสตร์,สังคม,300,300.0,0.0


---
## 6. Ranking Functions

กลุ่มฟังก์ชันจัดอันดับแถวภายในแต่ละ partition ซึ่งต้องมี `ORDER BY` ใน `OVER()` เสมอ

| ฟังก์ชัน | พฤติกรรมเมื่อค่าเสมอกัน |
|---|---|
| `ROW_NUMBER()` | ให้เลขต่างกันเสมอ ไม่ซ้ำเด็ดขาด |
| `RANK()` | ให้อันดับเท่ากัน แล้ว**ข้าม**เลขถัดไป |
| `DENSE_RANK()` | ให้อันดับเท่ากัน แล้ว**ไม่ข้าม**เลขถัดไป |
| `NTILE(n)` | แบ่งเป็น n กลุ่มตามตำแหน่งเท่านั้น |

In [ ]:
# ตัวอย่างที่ 1: เทียบทั้ง 4 ฟังก์ชันในข้อมูลที่มีราคาซ้ำกัน
con.execute("INSERT INTO book VALUES (5, 'ระบบสารสนเทศ', 'เทคโนโลยี', 420, 1)")  # ราคาเท่ากับเล่ม 1 เพื่อสาธิตค่าเสมอกัน

result = run("""
SELECT title, price,
    ROW_NUMBER() OVER (ORDER BY price DESC) AS row_number,
    RANK()       OVER (ORDER BY price DESC) AS rank,
    DENSE_RANK() OVER (ORDER BY price DESC) AS dense_rank,
    NTILE(2)     OVER (ORDER BY price DESC) AS ntile_2
FROM book
WHERE category = 'เทคโนโลยี';
""")
con.execute("DELETE FROM book WHERE book_id = 5")  # ลบแถวทดลองออก
result

,title,price,row_number,rank,dense_rank,ntile_2
0,ฐานข้อมูลเบื้องต้น,420,1,1,1,1
1,ระบบสารสนเทศ,420,2,1,1,1
2,วิทยาการข้อมูล,380,3,3,2,2


In [ ]:
# ตัวอย่างที่ 2: Top-N per group ด้วย ROW_NUMBER ใน CTE
# หาราคาที่สูงสุดในแต่ละกลุ่ม
run("""
WITH ranked AS (
    SELECT title, category, price,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS rn
    FROM book
)
SELECT title, category, price
FROM ranked r
WHERE rn = 1
ORDER BY category COLLATE th ASC;
""")

,title,category,price
0,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420
1,นิยายไทย,วรรณกรรม,250
2,ประวัติศาสตร์,สังคม,300


### แบบฝึกหัดที่ 6 (Ranking Functions)

**โจทย์:** ต้องการจัดกลุ่มสมาชิกตามความถี่การยืม — นับจำนวนครั้งที่แต่ละคนยืมหนังสือ แล้วแบ่งสมาชิกออกเป็น 3 กลุ่ม ด้วย NTILE(3)

In [ ]:
# เขียนโค้ดตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์

,name,total_loans,activity_tier
0,สมชาย,2,1
1,สมหญิง,2,1
2,มานะ,2,1
3,มานี,2,2
4,ปิติ,1,2
5,ชูใจ,1,2
6,วีระ,1,3
7,สมศรี,1,3


---
## 7. Aggregate Window & Frame

**Running Total** — ผลรวมสะสมตามลำดับ

**Window Frame** — กำหนดเองว่าจะนับแถวไหนบ้าง

In [ ]:
# ลองปริ้นข้อมูลสรุปยอดยืมรายเดือนออกมาดู
run("""
SELECT date_trunc('month', loan_date) AS mth, COUNT(*) AS loans
FROM loan
GROUP BY 1
ORDER BY 1;
""")

,mth,loans
0,2026-01-01,2
1,2026-02-01,3
2,2026-03-01,3
3,2026-04-01,2
4,2026-05-01,2


**Running Total** — ผลรวมสะสมตามลำดับ

In [ ]:
# หาผลรวมสะสมยอดการยืมต่อเนื่อง
run("""
WITH monthly AS (
    SELECT date_trunc('month', loan_date) AS mth, COUNT(*) AS loans
    FROM loan GROUP BY 1
)
SELECT mth, loans,
    SUM(loans) OVER (ORDER BY mth) AS running_total
FROM monthly ORDER BY mth;
""")

,mth,loans,running_total
0,2026-01-01,2,2.0
1,2026-02-01,3,5.0
2,2026-03-01,3,8.0
3,2026-04-01,2,10.0
4,2026-05-01,2,12.0


**Window Frame** — กำหนดเองว่าจะนับแถวไหนบ้าง

In [ ]:
# หาผลรวมเฉลี่ยของ 3 เดือนล่าสุด
run("""
WITH monthly AS (
    SELECT date_trunc('month', loan_date) AS mth, COUNT(*) AS loans
    FROM loan GROUP BY 1
)
SELECT mth, loans,
    ROUND(AVG(loans) OVER (
        ORDER BY mth
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3m
FROM monthly;
""")

,mth,loans,moving_avg_3m
0,2026-01-01,2,2.00
1,2026-02-01,3,2.50
2,2026-03-01,3,2.67
3,2026-04-01,2,2.67
4,2026-05-01,2,2.33


### แบบฝึกหัดที่ 7 (หัวข้อ Aggregate Window & Frame)

**โจทย์:** เขียน query แสดง running total และ moving average ของ 3 เดือน (เดือนก่อน + เดือนนี้ + เดือนถัดไป) ของยอดยืมรายเดือน

In [ ]:
# เขียนโค้ดตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์

,mth,loans,running_total,centered_moving_avg
0,2026-01-01,2,2.0,2.50
1,2026-02-01,3,5.0,2.67
2,2026-03-01,3,8.0,2.67
3,2026-04-01,2,10.0,2.33
4,2026-05-01,2,12.0,2.00


---
## 8. LAG/LEAD & Value Functions

- **`LAG(col)`** — ดึงค่าของแถว**ก่อนหน้า**มาไว้ในแถวปัจจุบัน
- **`LEAD(col)`** — ดึงค่าของแถว**ถัดไป**มาไว้ในแถวปัจจุบัน
- **`FIRST_VALUE(col)`** — ดึงค่า**แรกสุด**ของหน้าต่าง
- **`LAST_VALUE(col)`** — ดึงค่า**สุดท้าย**ของหน้าต่าง ต้องระบุ (`ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`)

In [ ]:
# ตัวอย่างที่ 1: LAG -- เทียบยอดกับเดือนก่อนหน้า
run("""
WITH monthly AS (
    SELECT date_trunc('month', loan_date) AS mth, COUNT(*) AS loans
    FROM loan GROUP BY 1
)
SELECT mth, loans,
    LAG(loans) OVER (ORDER BY mth) AS prev_month,
    loans - LAG(loans) OVER (ORDER BY mth) AS change
FROM monthly ORDER BY mth;
""")

,mth,loans,prev_month,change
0,2026-01-01,2,<NA>,<NA>
1,2026-02-01,3,2,1
2,2026-03-01,3,3,0
3,2026-04-01,2,3,-1
4,2026-05-01,2,2,0


In [ ]:
# ตัวอย่างที่ 2: LEAD -- เทียบยอดกับเดือนถัดไป
run("""
WITH monthly AS (
    SELECT date_trunc('month', loan_date) AS mth, COUNT(*) AS loans
    FROM loan GROUP BY 1
)
SELECT mth, loans,
    LEAD(loans) OVER (ORDER BY mth) AS next_month
FROM monthly ORDER BY mth;
""")

,mth,loans,next_month
0,2026-01-01,2,3
1,2026-02-01,3,3
2,2026-03-01,3,2
3,2026-04-01,2,2
4,2026-05-01,2,<NA>


In [ ]:
# ต้องการหาชื่อหนังสือแพงสุดในแต่ละหมวดคือเล่มไหน แล้วเติมชื่อเล่มนั้นลงไปทุกแถวในหมวดเดียวกัน
run("""
SELECT title, category, price,
    FIRST_VALUE(title) OVER (
        PARTITION BY category
        ORDER BY price DESC)
    AS most_expensive
FROM book
ORDER BY category COLLATE th ASC;
""")

,title,category,price,most_expensive
0,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,ฐานข้อมูลเบื้องต้น
1,วิทยาการข้อมูล,เทคโนโลยี,380,ฐานข้อมูลเบื้องต้น
2,นิยายไทย,วรรณกรรม,250,นิยายไทย
3,ประวัติศาสตร์,สังคม,300,ประวัติศาสตร์


อธิบายการทำงาน:

* `PARTITION BY category` แบ่งข้อมูลเป็นหน้าต่างย่อยตามหมวด (เทคโนโลยี 2 เล่ม, วรรณกรรม 1 เล่ม, สังคม 1 เล่ม)
* `ORDER BY price DESC` กำหนดว่า "ตำแหน่งแรกสุด" ของแต่ละหน้าต่างคือเล่มที่ราคาแพงที่สุด
* `FIRST_VALUE(title)` ดึงชื่อเล่มที่อยู่ตำแหน่งแรกมาเติมซ้ำลงทุกแถวในหน้าต่างนั้น — หมวดเทคโนโลยีทั้ง 2 แถวจึงได้ most_expensive = "ฐานข้อมูลเบื้องต้น" เหมือนกัน (420 บาท แพงกว่า 380)

In [ ]:
# ต้องการหาชื่อหนังสือถูกสุดในแต่ละหมวดคือเล่มไหน แล้วเติมชื่อเล่มนั้นลงไปทุกแถวในหมวดเดียวกัน
run("""
SELECT title, category, price,
    LAST_VALUE(title) OVER (
        PARTITION BY category ORDER BY price DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS cheapest_correct
FROM book
ORDER BY category COLLATE th ASC;
""")

,title,category,price,cheapest_correct
0,ฐานข้อมูลเบื้องต้น,เทคโนโลยี,420,วิทยาการข้อมูล
1,วิทยาการข้อมูล,เทคโนโลยี,380,วิทยาการข้อมูล
2,นิยายไทย,วรรณกรรม,250,นิยายไทย
3,ประวัติศาสตร์,สังคม,300,ประวัติศาสตร์


### แบบฝึกหัดที่ 8 (LAG/LEAD & Value Functions)

**โจทย์:** สำหรับสมาชิกแต่ละคน จงเขียน query แสดง

1. วันที่ยืมครั้งแรกของสมาชิกคนนั้น (first_loan)
2. วันที่ยืมครั้งล่าสุดของสมาชิกคนนั้น (last_loan)
3. คอลัมน์บอกว่าแถวนี้คือ "ยืมครั้งแรก" ของสมาชิกคนนั้นหรือไม่ (is_first_time) ซึ่งต้องการเฉพาะที่ไม่ใช่การยืมครั้งแรกของสมาชิกคนนั้น

In [ ]:
# เขียนโค้ดตรงนี้
run("""

""")

In [ ]:
# ตัวอย่างผลลัพธ์

,name,loan_date,first_loan,last_loan,is_first_time
0,มานะ,2026-03-01,2026-02-15,2026-03-01,ไม่ใช่
1,มานี,2026-03-20,2026-03-08,2026-03-20,ไม่ใช่
2,สมชาย,2026-02-02,2026-01-05,2026-02-02,ไม่ใช่
3,สมหญิง,2026-02-10,2026-01-20,2026-02-10,ไม่ใช่


---
## จบ Chapter 5 แล้ว